In [2]:
!pip install xgboost


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix,hstack
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

In [4]:
df = pd.read_csv(r"Feature Engineered IMDB Dataset.csv")

In [5]:
df.head()

,Unnamed: 0.1,Unnamed: 0,review,sentiment,cleaned_review,tokenized_review,stemmed review,filtered review,feature word count,sentence length,feature punctuation frequency,feature readability,most_frequent_terms,feature_real_word_ratio
0,0,0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,"['one', 'of', 'the', 'other', 'reviewers', 'ha...","['one', 'of', 'the', 'other', 'review', 'has',...","['review', 'mention', 'watch', '1', 'oz', 'epi...",307,1761,43,66.152200,"['oz', 'violenc', 'watch']",0.832237
1,1,1,A wonderful little production. <br /><br />The...,positive,a wonderful little production the filming tec...,"['a', 'wonderful', 'little', 'production', 'th...","['a', 'wonder', 'littl', 'product', 'the', 'fi...","['wonder', 'littl', 'product', 'film', 'techni...",162,998,16,45.478333,"['littl', 'product', 'techniqu']",0.826923
2,2,2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,"['i', 'thought', 'this', 'was', 'a', 'wonderfu...","['i', 'thought', 'this', 'was', 'a', 'wonder',...","['thought', 'wonder', 'way', 'spend', 'time', ...",166,926,23,57.459357,"['comedi', 'thought', 'woodi']",0.908537
3,3,3,Basically there's a family where a little boy ...,negative,basically theres a family where a little boy j...,"['basically', 'theres', 'a', 'family', 'where'...","['basic', 'there', 'a', 'famili', 'where', 'a'...","['basic', 'famili', 'littl', 'boy', 'jake', 't...",138,748,24,75.319167,"['jake', 'parent', 'movi']",0.863636
4,4,4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love in the time of money is a ...,"['petter', 'matteis', 'love', 'in', 'the', 'ti...","['petter', 'mattei', 'love', 'in', 'the', 'tim...","['petter', 'mattei', 'love', 'time', 'money', ...",230,1317,34,60.138177,"['mattei', 'mr', 'peopl']",0.893333


In [6]:
# Cleaning up the index 
df = df.drop(columns = ['Unnamed: 0', 'Unnamed: 0.1'],errors = 'ignore')

In [7]:
# Converting the target (sentiment) [positive/negative] to numeric [1/0]
df['Sentiment Label'] = df['sentiment'].map({'positive':1,'negative':0})

In [8]:
# Defining out text column and our engineered numeric column
text_col = 'cleaned_review'
numeric_cols = ['feature word count',
 'sentence length',
 'feature punctuation frequency',
 'feature readability',
 'feature_real_word_ratio']

In [9]:
df[text_col]

0        one of the other reviewers has mentioned that ...
1        a wonderful little production  the filming tec...
2        i thought this was a wonderful way to spend ti...
3        basically theres a family where a little boy j...
4        petter matteis love in the time of money is a ...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot, bad dialogue, bad acting, idiotic di...
49997    i am a catholic taught in parochial elementary...
49998    i'm going to have to disagree with the previou...
49999    no one expects the star trek movies to be high...
Name: cleaned_review, Length: 50000, dtype: object

In [10]:
df[numeric_cols]

,feature word count,sentence length,feature punctuation frequency,feature readability,feature_real_word_ratio
0,307,1761,43,66.152200,0.832237
1,162,998,16,45.478333,0.826923
2,166,926,23,57.459357,0.908537
3,138,748,24,75.319167,0.863636
4,230,1317,34,60.138177,0.893333
...,...,...,...,...,...
49995,194,1008,14,70.702503,0.814159
49996,112,642,19,59.465071,0.790323
49997,230,1280,22,61.205855,0.824903
49998,212,1234,23,54.234670,0.770833


In [11]:
X_text = df[text_col]
X_numeric = df[numeric_cols]
y = df['Sentiment Label']

In [12]:
# Train-Validation-Test Split
# Splitting into 70% train and 30% remaining dataset
X_text_train, X_text_rem, X_num_train, X_num_rem, y_train, y_rem = train_test_split(X_text, X_numeric, y, test_size = 0.30, random_state=42, stratify = y)

In [13]:
# splitting remaining 30% into validation (15%) and test (15%)
X_text_val, X_text_test, X_num_val, X_num_test, y_val, y_test = train_test_split(X_text_rem, X_num_rem, y_rem, test_size = 0.50, random_state = 42, stratify = y_rem)

In [14]:
# Transformation and Feature Scaling

In [15]:
# TF-IDF on Text
tfidf = TfidfVectorizer(max_features = 5000)
X_train_tfidf = tfidf.fit_transform(X_text_train)
X_val_tfidf = tfidf.transform(X_text_val)
X_test_tfidf = tfidf.transform(X_text_test)

In [16]:
# Scaling numeric engineered features (Important for LR and SVM)
scaler = MinMaxScaler()
X_train_num_scaled = scaler.fit_transform(X_num_train)
X_val_num_scaled = scaler.transform(X_num_val)
X_test_num_scaled = scaler.transform(X_num_test)

In [17]:
# Combinig Text (TF-IDF) and Numeric Engineered Features
X_train_combined = hstack([X_train_tfidf, X_train_num_scaled]).toarray()
X_val_combined = hstack([X_val_tfidf, X_val_num_scaled]).toarray()
X_test_combined = hstack([X_test_tfidf, X_test_num_scaled]).toarray()

In [18]:
# Converting the scaled dense numeric features to a sparse matrix format
X_train_num_sparse = csr_matrix(X_train_num_scaled)
X_val_num_sparse = csr_matrix(X_val_num_scaled)   
X_test_num_sparse = csr_matrix(X_test_num_scaled)

# Stacking them together (Keep them completely sparse!)
X_train_combined = hstack([X_train_tfidf, X_train_num_sparse])
X_val_combined = hstack([X_val_tfidf, X_val_num_sparse])
X_test_combined = hstack([X_test_tfidf, X_test_num_sparse])

In [19]:
# Models 
models = {
    "Logistic Regression": LogisticRegression(max_iter = 1000, random_state = 42),
    "Naive Bayes": MultinomialNB(),
    "SVM":LinearSVC(dual='auto',random_state = 42),
    "Random Forest": RandomForestClassifier(random_state = 42),
    "XGBoost": XGBClassifier(use_label_encoder = False, eval_metrics = 'logloss', random_state = 42)
}

In [20]:
results_list = []

for name, model in models.items():
    # Train model on the combined feature dataset
    model.fit(X_train_combined, y_train)
    
    # Predict on Validation data
    y_pred = model.predict(X_val_combined)
    
    # Calculate performance metrics
    accuracy = accuracy_score(y_val, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_val, y_pred, average='binary', zero_division=0)
    cm = confusion_matrix(y_val, y_pred)
    
    # Save validation metrics for comparison
    results_list.append({
        "Model": name,
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4)
    })
    
    print(f"\n{name} Confusion Matrix")
    print(cm)


Logistic Regression Confusion Matrix
[[3314  436]
 [ 401 3349]]

Naive Bayes Confusion Matrix
[[3183  567]
 [ 566 3184]]

SVM Confusion Matrix
[[3300  450]
 [ 436 3314]]

Random Forest Confusion Matrix
[[3182  568]
 [ 614 3136]]


c:\Users\rashm\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:34:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "eval_metrics", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost Confusion Matrix
[[3140  610]
 [ 480 3270]]


In [21]:
# MODEL COMPARISON TABLE
df_comparison = pd.DataFrame(results_list)
print("\nMODEL COMPARISON TABLE (Validation Set)")
print(df_comparison.to_string(index=False))


MODEL COMPARISON TABLE (Validation Set)
              Model  Accuracy  Precision  Recall  F1-Score
Logistic Regression    0.8884     0.8848  0.8931    0.8889
        Naive Bayes    0.8489     0.8488  0.8491    0.8490
                SVM    0.8819     0.8804  0.8837    0.8821
      Random Forest    0.8424     0.8467  0.8363    0.8414
            XGBoost    0.8547     0.8428  0.8720    0.8571


In [22]:
df.to_csv('Feature Engineered IMDB Dataset - 2.csv')